# Feature Engineering and Baseline ML (WEEK 2)

In [7]:
# Setup & Installs
# !pip install xgboost pandas numpy scikit-learn
!pip install xgboost

   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   - -------------------------------------- 2.6/69.5 MB 18.0 MB/s eta 0:00:04
   -- ------------------------------------- 4.5/69.5 MB 13.5 MB/s eta 0:00:05
   --- ------------------------------------ 5.8/69.5 MB 9.7 MB/s eta 0:00:07
   --- ------------------------------------ 6.3/69.5 MB 8.7 MB/s eta 0:00:08
   --- ------------------------------------ 6.8/69.5 MB 6.5 MB/s eta 0:00:10
   ---- ----------------------------------- 7.3/69.5 MB 5.7 MB/s eta 0:00:11
   ---- ----------------------------------- 7.9/69.5 MB 5.3 MB/s eta 0:00:12
   ---- ----------------------------------- 8.4/69.5 MB 5.1 MB/s eta 0:00:13
   ----- ---------------------------------- 9.2/69.5 MB 4.9 MB/s eta 0:00:13
   ----- ---------------------------------- 10.2/69.5 MB 4.8 MB/s eta 0:00:13
   ------ --------------------------------- 11.0/69.5 MB 4.8 MB/s eta 0:00:13
   ------ --------------------------------- 11.8/69.5 MB 4.7 MB/s eta 0:00:13
 

In [ ]:
import pandas as pd
import numpy as np
from math import radians, cos, sin, asin, sqrt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import xgboost as xgb

df = pd.read_csv('AB_NYC_2019.csv')

### 1. Feature Engineering
Calculating the Haversine distance from each property to the approximate center of New York City (Empire State Building).

In [ ]:
def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371 # Radius of Earth in kilometers
    return c * r

city_center_lat = 40.748817
city_center_lon = -73.985428

df['distance_to_center_km'] = df.apply(lambda row: haversine(city_center_lon, city_center_lat, row['longitude'], row['latitude']), axis=1)

features = ['neighbourhood_group', 'room_type', 'minimum_nights', 'number_of_reviews', 'calculated_host_listings_count', 'availability_365', 'distance_to_center_km']
X = df[features]
y = df['price']

numeric_features = ['minimum_nights', 'number_of_reviews', 'calculated_host_listings_count', 'availability_365', 'distance_to_center_km']
categorical_features = ['neighbourhood_group', 'room_type']

# For missing values
for col in numeric_features:
    X.loc[:, col] = X[col].fillna(X[col].median())
for col in categorical_features:
    X.loc[:, col] = X[col].fillna(X[col].mode()[0])

### 2. Model Pipeline Setup

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numeric_features)
    ])

# Train Baseline XGBoost Model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRegressor(objective='reg:squarederror', random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['neighbourhood_group',
                                                   'room_type']),
                                                 ('num', 'passthrough',
                                                  ['minimum_nights',
                                                   'number_of_reviews',
                                                   'calculated_host_listings_count',
                                                   'availability_365',
                                                   'distance_to_center_km'])])),
                ('regressor',
                 XGBRegressor(base_score=None, booster=None, callback...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=None,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=None, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=None, n_jobs=None,
                              num_parallel_tree=None, ...))])

### 3. Evaluation Metrics

In [11]:
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = mean_absolute_percentage_error(y_test, y_pred)

print(f"Baseline RMSE: {rmse:.2f}")
print(f"Baseline MAPE: {mape:.2f}")

Baseline RMSE: 217.89
Baseline MAPE: 94311366524928.00
